In [ ]:
# Cell 1: Imports and setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import gc

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# CPU only for now — no .to(device) / CUDA calls in Weeks 1-5
print("PyTorch version:", torch.__version__)
print("Running on CPU")

PyTorch version: 2.11.0+cpu
Running on CPU


In [ ]:
# ============================================================
# HARDWARE MONITORING
# ============================================================

!pip install -q psutil nvidia-ml-py


import os
import time
import threading
import numpy as np
import pandas as pd
import psutil



In [ ]:
# ------------------------------------------------------------
# NVIDIA NVML
# ------------------------------------------------------------

try:
    import pynvml

    pynvml.nvmlInit()
    NVML_AVAILABLE = True

except Exception:
    NVML_AVAILABLE = False

In [ ]:
# ------------------------------------------------------------
# Hardware Monitor
# ------------------------------------------------------------

class HardwareMonitor:

    def __init__(self, interval=0.2, gpu_index=0):

        self.interval = interval
        self.gpu_index = gpu_index

        self.running = False
        self.thread = None

        # CPU
        self.cpu_util_samples = []
        self.ram_samples = []

        # GPU
        self.gpu_util_samples = []
        self.gpu_mem_util_samples = []
        self.vram_samples = []
        self.gpu_power_samples = []
        self.gpu_temp_samples = []

        self.timestamps = []

        self.process = psutil.Process(os.getpid())

        self.gpu_handle = None

        if NVML_AVAILABLE:

            try:
                self.gpu_handle = (
                    pynvml.nvmlDeviceGetHandleByIndex(
                        gpu_index
                    )
                )

            except Exception:
                self.gpu_handle = None

    # --------------------------------------------------------
    # Start
    # --------------------------------------------------------
    def start(self):

        # Initialize process CPU counter
        self.process.cpu_percent(None)

        self.running = True

        self.thread = threading.Thread(
            target=self._monitor,
            daemon=True
        )

        self.thread.start()

    # --------------------------------------------------------
    # Monitoring loop
    # --------------------------------------------------------
    def _monitor(self):

        logical_cpus = psutil.cpu_count(
            logical=True
        )

        while self.running:

            timestamp = time.perf_counter()
            # =================================================
            # CPU UTILIZATION
            # =================================================
            process_cpu = (
                self.process.cpu_percent(
                    interval=None
                )
            )

            # Normalize process CPU usage to
            # percentage of total logical CPU capacity.
            if logical_cpus:
                process_cpu_normalized = (
                    process_cpu / logical_cpus
                )
            else:
                process_cpu_normalized = process_cpu
            self.cpu_util_samples.append(
                process_cpu_normalized
            )

            # =================================================
            # RAM
            # =================================================

            ram_mb = (
                self.process.memory_info().rss
                / (1024 ** 2)
            )

            self.ram_samples.append(
                ram_mb
            )

            # =================================================
            # GPU
            # =================================================

            if self.gpu_handle is not None:

                try:

                    utilization = (
                        pynvml.nvmlDeviceGetUtilizationRates(
                            self.gpu_handle
                        )
                    )

                    memory = (
                        pynvml.nvmlDeviceGetMemoryInfo(
                            self.gpu_handle
                        )
                    )

                    power = (
                        pynvml.nvmlDeviceGetPowerUsage(
                            self.gpu_handle
                        ) / 1000.0
                    )

                    temperature = (
                        pynvml.nvmlDeviceGetTemperature(
                            self.gpu_handle,
                            pynvml.NVML_TEMPERATURE_GPU
                        )
                    )

                    # GPU compute utilization
                    self.gpu_util_samples.append(
                        float(utilization.gpu)
                    )

                    # GPU memory-controller utilization
                    self.gpu_mem_util_samples.append(
                        float(utilization.memory)
                    )

                    # VRAM used
                    self.vram_samples.append(
                        memory.used / (1024 ** 2)
                    )

                    # Power in Watts
                    self.gpu_power_samples.append(
                        power
                    )

                    # Temperature
                    self.gpu_temp_samples.append(
                        float(temperature)
                    )

                except Exception:
                    pass

            self.timestamps.append(
                timestamp
            )

            time.sleep(
                self.interval
            )

    # --------------------------------------------------------
    # Stop
    # --------------------------------------------------------

    def stop(self):

        self.running = False

        if self.thread is not None:
            self.thread.join()

    # --------------------------------------------------------
    # Results
    # --------------------------------------------------------

    def get_results(self):

        result = {}

        # =====================================================
        # CPU
        # =====================================================

        result["avg_cpu_util_percent"] = (
            np.mean(
                self.cpu_util_samples
            )
            if self.cpu_util_samples
            else np.nan
        )

        result["peak_cpu_util_percent"] = (
            np.max(
                self.cpu_util_samples
            )
            if self.cpu_util_samples
            else np.nan
        )

        result["avg_ram_mb"] = (
            np.mean(
                self.ram_samples
            )
            if self.ram_samples
            else np.nan
        )

        result["peak_ram_mb"] = (
            np.max(
                self.ram_samples
            )
            if self.ram_samples
            else np.nan
        )

        # =====================================================
        # GPU
        # =====================================================

        if self.gpu_util_samples:

            result["avg_gpu_util_percent"] = np.mean(
                self.gpu_util_samples
            )

            result["peak_gpu_util_percent"] = np.max(
                self.gpu_util_samples
            )

            result["avg_gpu_memory_util_percent"] = np.mean(
                self.gpu_mem_util_samples
            )

            result["peak_gpu_memory_util_percent"] = np.max(
                self.gpu_mem_util_samples
            )

            result["avg_vram_mb"] = np.mean(
                self.vram_samples
            )

            result["peak_vram_mb"] = np.max(
                self.vram_samples
            )

            result["avg_gpu_power_w"] = np.mean(
                self.gpu_power_samples
            )

            result["peak_gpu_power_w"] = np.max(
                self.gpu_power_samples
            )

            result["avg_gpu_temperature_c"] = np.mean(
                self.gpu_temp_samples
            )

            result["peak_gpu_temperature_c"] = np.max(
                self.gpu_temp_samples
            )

            # =================================================
            # Energy
            # E = integral(P dt)
            # Trapezoidal approximation
            # =================================================

            energy_joules = 0.0

            n = min(
                len(self.gpu_power_samples),
                len(self.timestamps)
            )

            for i in range(1, n):

                dt = (
                    self.timestamps[i]
                    - self.timestamps[i - 1]
                )

                avg_power = (
                    self.gpu_power_samples[i]
                    + self.gpu_power_samples[i - 1]
                ) / 2.0

                energy_joules += (
                    avg_power * dt
                )

            result["gpu_energy_joules"] = (
                energy_joules
            )

        else:

            result["avg_gpu_util_percent"] = np.nan
            result["peak_gpu_util_percent"] = np.nan

            result["avg_gpu_memory_util_percent"] = np.nan
            result["peak_gpu_memory_util_percent"] = np.nan

            result["avg_vram_mb"] = np.nan
            result["peak_vram_mb"] = np.nan

            result["avg_gpu_power_w"] = np.nan
            result["peak_gpu_power_w"] = np.nan

            result["avg_gpu_temperature_c"] = np.nan
            result["peak_gpu_temperature_c"] = np.nan

            result["gpu_energy_joules"] = np.nan

        return result

In [ ]:
# ------------------------------------------------------------
# Save 5-run results
# ------------------------------------------------------------

def save_five_run_results(
    run_results,
    filename,
    algorithm,
    device
):

    rows = []

    for result in run_results:

        row = result.copy()

        row["Algorithm"] = algorithm
        row["Device"] = device

        rows.append(row)

    runs_df = pd.DataFrame(rows)

    # --------------------------------------------------------
    # Calculate average
    # --------------------------------------------------------

    numeric_columns = [
        c for c in runs_df.columns
        if c not in ["run"]
        and pd.api.types.is_numeric_dtype(
            runs_df[c]
        )
    ]

    average_row = {
        "run": "AVERAGE",
        "Algorithm": algorithm,
        "Device": device
    }

    for column in numeric_columns:

        average_row[column] = (
            runs_df[column].mean()
        )

    # --------------------------------------------------------
    # Calculate standard deviation
    # --------------------------------------------------------

    std_row = {
        "run": "STD",
        "Algorithm": algorithm,
        "Device": device
    }

    for column in numeric_columns:

        std_row[column] = (
            runs_df[column].std()
        )

    final_df = pd.concat(
        [
            runs_df,
            pd.DataFrame([
                average_row,
                std_row
            ])
        ],
        ignore_index=True
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    final_df.to_csv(
        filename,
        index=False
    )

    print("\n" + "=" * 80)
    print(f"{algorithm} - {device}")
    print("=" * 80)

    display(
        final_df.round(3)
    )

    print(
        f"\nSaved result file:\n{filename}"
    )

    return final_df

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

RESULT_DIR = (
    "/content/drive/MyDrive/"
    "GCMC_Project/hardware_results"
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

print(
    "Results directory:",
    RESULT_DIR
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results directory: /content/drive/MyDrive/GCMC_Project/hardware_results


In [ ]:
# Cell 2: Load the ratings dataset
df = pd.read_csv("ratings.csv")

print("Shape:", df.shape)
print("\nUsers :", df.userId.nunique())
print("Movies:", df.movieId.nunique())

print("\nRating values:")
print(sorted(df.rating.unique()))

df.head()

Shape: (100836, 4)

Users : 610
Movies: 9724

Rating values:
[np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
# Cell 3: Map ratings to class indices, and build user/movie index mappings

# Ratings are things like 0.5, 1.0, 1.5, ... 5.0 -> we map each to a class index 0..N-1
rating_values = sorted(df.rating.unique())
rating_to_class = {r: i for i, r in enumerate(rating_values)}
class_to_rating = {i: r for i, r in enumerate(rating_values)}

print("Rating classes:", rating_to_class)
print("Number of classes:", len(rating_values))

# Map original userId/movieId to sequential indices (0 to N-1)
user_map = {u: i for i, u in enumerate(df.userId.unique())}
movie_map = {m: i for i, m in enumerate(df.movieId.unique())}

df["user_idx"] = df.userId.map(user_map)
df["movie_idx"] = df.movieId.map(movie_map)

NUM_USERS = len(user_map)
NUM_MOVIES = len(movie_map)
NUM_NODES = NUM_USERS + NUM_MOVIES

print("\nUsers :", NUM_USERS)
print("Movies:", NUM_MOVIES)
print("Nodes :", NUM_NODES)

df.head()

Rating classes: {np.float64(0.5): 0, np.float64(1.0): 1, np.float64(1.5): 2, np.float64(2.0): 3, np.float64(2.5): 4, np.float64(3.0): 5, np.float64(3.5): 6, np.float64(4.0): 7, np.float64(4.5): 8, np.float64(5.0): 9}
Number of classes: 10

Users : 610
Movies: 9724
Nodes : 10334


,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,1,4.0,964982703,0,0
1,1,3,4.0,964981247,0,1
2,1,6,4.0,964982224,0,2
3,1,47,5.0,964983815,0,3
4,1,50,5.0,964982931,0,4


In [ ]:
# Cell 4: Train/test split (no subsampling — using the full 100k dataset)
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print("Train size:", len(train_df))
print("Test size :", len(test_df))

Train size: 80668
Test size : 20168


In [ ]:
# Cell 5: Build edges for the GCMC bipartite user-movie graph
# Users occupy node indices [0, NUM_USERS), movies occupy [NUM_USERS, NUM_NODES)

edge_dict = {r: [] for r in rating_values}

for row in train_df.itertuples():
    user_node = row.user_idx
    movie_node = NUM_USERS + row.movie_idx

    # Bi-directional edges (user->movie and movie->user) for message passing
    edge_dict[row.rating].append([user_node, movie_node])
    edge_dict[row.rating].append([movie_node, user_node])

for r in rating_values:
    print(r, "->", len(edge_dict[r]), "edges")

0.5 -> 2162 edges
1.0 -> 4510 edges
1.5 -> 2798 edges
2.0 -> 12036 edges
2.5 -> 8940 edges
3.0 -> 32136 edges
3.5 -> 20976 edges
4.0 -> 43014 edges
4.5 -> 13638 edges
5.0 -> 21126 edges


In [ ]:
# Convert edge lists into arrays PyTorch can use (CPU, plain torch.long)
edge_index_dict = {}
for r in rating_values:
    edges = edge_dict[r]
    if len(edges) == 0:
        edge_index_dict[r] = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index_dict[r] = torch.tensor(edges, dtype=torch.long).t()

total_edges = sum(edge_index_dict[r].shape[1] for r in rating_values)
print(f"\nTotal edges in graph: {total_edges}")


Total edges in graph: 161336


In [ ]:
# Cell 6: GCMC Encoder — builds node representations from the graph

class GCMCEncoder(nn.Module):
    def __init__(self, num_nodes, hidden_dim):
        super().__init__()

        # Embedding layer holds a learnable vector for every user and movie node
        self.embedding = nn.Embedding(num_nodes, hidden_dim)

        # One linear layer per rating type, used during message passing
        self.rating_layers = nn.ModuleDict()
        for c in range(len(rating_values)):
            self.rating_layers[str(c)] = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def propagate(self, x, edge_index, linear):
        # No edges for this rating type -> nothing to aggregate
        if edge_index.shape[1] == 0:
            return torch.zeros_like(x)

        src = edge_index[0]
        dst = edge_index[1]

        # Transform the source node's vector, then send it to the destination node
        messages = linear(x[src])

        out = torch.zeros_like(x)
        out.index_add_(0, dst, messages)  # sum incoming messages per destination node
        return out

    def forward(self, edge_index_dict):
        x = self.embedding.weight  # starting vectors for all nodes

        aggregate = torch.zeros_like(x)
        for c, r in enumerate(rating_values):
            aggregate += self.propagate(x, edge_index_dict[r], self.rating_layers[str(c)])

        return F.relu(aggregate)

In [ ]:
# Cell 7: Quick sanity check — run the encoder once and inspect output shape
encoder = GCMCEncoder(NUM_NODES, hidden_dim=16)

embeddings = encoder(edge_index_dict)

print("Embeddings shape:", embeddings.shape)
# Expect: (NUM_NODES, 16) -> one 16-dim vector per user/movie node

Embeddings shape: torch.Size([10334, 16])


In [ ]:
# Cell 8: Bilinear Decoder — turns (user, movie) embeddings into a rating probability distribution

class BilinearDecoder(nn.Module):
    def __init__(self, hidden_dim, num_classes):
        super().__init__()

        # One learnable Q matrix per rating class
        self.Q = nn.ParameterList()
        for _ in range(num_classes):
            self.Q.append(nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01))

    def forward(self, user_emb, movie_emb):
        scores = []

        # For each rating class, compute a bilinear score: user_emb^T . Q . movie_emb
        for q in self.Q:
            score = torch.sum((user_emb @ q) * movie_emb, dim=1)
            scores.append(score)

        scores = torch.stack(scores, dim=1)     # shape: (batch, num_classes)
        probs = F.softmax(scores, dim=1)        # convert to a probability distribution
        return probs

In [ ]:
# Quick sanity check
decoder = BilinearDecoder(hidden_dim=16, num_classes=len(rating_values))

dummy_user = embeddings[:5]
dummy_movie = embeddings[5:10]

probs = decoder(dummy_user, dummy_movie)
print("Probs shape:", probs.shape)         # (5, num_classes)
print("Sum of probs for user 0:", probs[0].sum().item())  # should be ~1.0

Probs shape: torch.Size([5, 10])
Sum of probs for user 0: 1.0


In [ ]:
# Cell 9: Full GCMC model — wraps the encoder and decoder together

class GCMC(nn.Module):
    def __init__(self, num_nodes, hidden_dim, num_classes):
        super().__init__()
        self.encoder = GCMCEncoder(num_nodes, hidden_dim)
        self.decoder = BilinearDecoder(hidden_dim, num_classes)

    def forward(self, edge_index_dict, users, movies):
        embeddings = self.encoder(edge_index_dict)

        user_emb = embeddings[users]
        movie_emb = embeddings[movies]

        probs = self.decoder(user_emb, movie_emb)
        return probs

In [ ]:
# Quick sanity check with a fresh model instance
model = GCMC(NUM_NODES, hidden_dim=16, num_classes=len(rating_values))

with torch.no_grad():
    sample_probs = model(edge_index_dict, users=torch.tensor([0, 1, 2]), movies=torch.tensor([NUM_USERS, NUM_USERS+1, NUM_USERS+2]))

print("Sample output shape:", sample_probs.shape)  # (3, num_classes)

Sample output shape: torch.Size([3, 10])


In [ ]:
# Cell 10: Prepare training data arrays from train_df

train_users = torch.tensor(
    train_df.userId.map(user_map).values,
    dtype=torch.long
)

train_movies = torch.tensor(
    NUM_USERS + train_df.movieId.map(movie_map).values,
    dtype=torch.long
)

train_labels = torch.tensor(
    train_df.rating.map(rating_to_class).values,
    dtype=torch.long
)

print("train_users :", train_users.shape)
print("train_movies:", train_movies.shape)
print("train_labels:", train_labels.shape)

train_users : torch.Size([80668])
train_movies: torch.Size([80668])
train_labels: torch.Size([80668])


In [ ]:
# ============================================================
# GCMC CPU - 5 RUN HARDWARE EXPERIMENT
# ============================================================

import torch
import torch.nn.functional as F

N_RUNS = 5
EPOCHS = 20

gcmc_cpu_runs = []


for run in range(1, N_RUNS + 1):

    print(
        f"\n{'=' * 25}"
        f" GCMC CPU RUN {run}/5 "
        f"{'=' * 25}"
    )

    # --------------------------------------------------------
    # Fresh model for every run
    # --------------------------------------------------------

    torch.manual_seed(
        1000 + run
    )

    model = GCMC(
        NUM_NODES,
        hidden_dim=16,
        num_classes=len(rating_values)
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    # --------------------------------------------------------
    # Start monitoring
    # --------------------------------------------------------

    monitor = HardwareMonitor(
        interval=0.2
    )

    monitor.start()

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model.train()

    for epoch in range(EPOCHS):

        optimizer.zero_grad()

        probs = model(
            edge_index_dict,
            train_users,
            train_movies
        )

        loss = F.nll_loss(
            torch.log(
                probs + 1e-10
            ),
            train_labels
        )

        loss.backward()

        optimizer.step()

    # --------------------------------------------------------
    # Stop monitoring
    # --------------------------------------------------------

    elapsed = (
        time.perf_counter()
        - start_time
    )

    monitor.stop()

    result = monitor.get_results()

    result["run"] = run
    result["training_time_s"] = elapsed

    gcmc_cpu_runs.append(
        result
    )

    print(
        f"Training time: {elapsed:.4f} s"
    )


# ============================================================
# SAVE FINAL RESULTS
# ============================================================

gcmc_cpu_file = os.path.join(
    RESULT_DIR,
    "GCMC_CPU_hardware_results.csv"
)

gcmc_cpu_results_df = (
    save_five_run_results(
        gcmc_cpu_runs,
        gcmc_cpu_file,
        "GCMC",
        "CPU"
    )
)


========================= GCMC CPU RUN 1/5 =========================
Training time: 3.8616 s

========================= GCMC CPU RUN 2/5 =========================
Training time: 6.1702 s

========================= GCMC CPU RUN 3/5 =========================
Training time: 3.6774 s

========================= GCMC CPU RUN 4/5 =========================
Training time: 3.9568 s

========================= GCMC CPU RUN 5/5 =========================
Training time: 4.3890 s

GCMC - CPU


,avg_cpu_util_percent,peak_cpu_util_percent,avg_ram_mb,peak_ram_mb,avg_gpu_util_percent,peak_gpu_util_percent,avg_gpu_memory_util_percent,peak_gpu_memory_util_percent,avg_vram_mb,peak_vram_mb,avg_gpu_power_w,peak_gpu_power_w,avg_gpu_temperature_c,peak_gpu_temperature_c,gpu_energy_joules,run,training_time_s,Algorithm,Device
0,46.988,52.350,743.119,782.285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3.862,GCMC,CPU
1,35.839,52.350,811.185,834.816,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,6.170,GCMC,CPU
2,47.358,52.150,806.021,817.656,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3.677,GCMC,CPU
3,47.235,52.400,841.497,874.039,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,3.957,GCMC,CPU
4,47.132,52.350,841.186,874.164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,4.389,GCMC,CPU
5,44.910,52.320,808.602,836.592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AVERAGE,4.411,GCMC,CPU
6,5.073,0.097,40.141,39.131,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STD,1.018,GCMC,CPU



Saved result file:
/content/drive/MyDrive/GCMC_Project/hardware_results/GCMC_CPU_hardware_results.csv


In [ ]:
# Cell 12: Evaluate GCMC on the test set

test_users = torch.tensor(
    test_df.userId.map(user_map).values,
    dtype=torch.long
)

test_movies = torch.tensor(
    NUM_USERS + test_df.movieId.map(movie_map).values,
    dtype=torch.long
)

test_actual_ratings = test_df.rating.values

model.eval()
with torch.no_grad():
    probs_test = model(edge_index_dict, test_users, test_movies)

# Convert predicted probabilities into an expected rating (weighted sum over classes)
rating_values_arr = torch.tensor(rating_values, dtype=torch.float32)
preds_gcmc = (probs_test * rating_values_arr).sum(dim=1).numpy()

gcmc_rmse = np.sqrt(mean_squared_error(test_actual_ratings, preds_gcmc))

print("====================")
print("GCMC RESULTS")
print("====================")
print(f"RMSE = {gcmc_rmse:.4f}")

GCMC RESULTS
RMSE = 1.0630


In [ ]:
# Cell 13: SVD METHOD
# Prepare training data for FunkSVD (uses raw user/movie indices, not the graph node offset)

train_users_svd = torch.tensor(
    train_df.userId.map(user_map).values,
    dtype=torch.long
)

train_movies_svd = torch.tensor(
    train_df.movieId.map(movie_map).values,
    dtype=torch.long
)

train_ratings_svd = torch.tensor(
    train_df.rating.values,
    dtype=torch.float32
)

print("train_users_svd  :", train_users_svd.shape)
print("train_movies_svd :", train_movies_svd.shape)
print("train_ratings_svd:", train_ratings_svd.shape)

train_users_svd  : torch.Size([80668])
train_movies_svd : torch.Size([80668])
train_ratings_svd: torch.Size([80668])


In [ ]:
# Cell 14: FunkSVD model — classic matrix factorization with biases

class FunkSVD(nn.Module):
    def __init__(self, num_users, num_movies, latent_dim=50):
        super().__init__()

        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.movie_embedding = nn.Embedding(num_movies, latent_dim)

        self.user_bias = nn.Embedding(num_users, 1)
        self.movie_bias = nn.Embedding(num_movies, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.movie_embedding.weight, std=0.01)

    def forward(self, users, movies):
        p = self.user_embedding(users)
        q = self.movie_embedding(movies)

        interaction = (p * q).sum(dim=1)

        pred = (
            interaction
            + self.user_bias(users).squeeze()
            + self.movie_bias(movies).squeeze()
            + self.global_bias
        )
        return pred

In [ ]:
# ============================================================
# FunkSVD CPU - 5 RUN HARDWARE EXPERIMENT
# ============================================================

N_RUNS = 5
EPOCHS = 30

svd_cpu_runs = []


for run in range(1, N_RUNS + 1):

    print(
        f"\n{'=' * 25}"
        f" FunkSVD CPU RUN {run}/5 "
        f"{'=' * 25}"
    )

    # --------------------------------------------------------
    # Fresh model
    # --------------------------------------------------------

    torch.manual_seed(
        2000 + run
    )

    svd_model = FunkSVD(
        NUM_USERS,
        NUM_MOVIES,
        latent_dim=50
    )

    optimizer = torch.optim.Adam(
        svd_model.parameters(),
        lr=0.005
    )

    loss_fn = nn.MSELoss()

    # --------------------------------------------------------
    # Monitor
    # --------------------------------------------------------

    monitor = HardwareMonitor(
        interval=0.2
    )

    monitor.start()

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(EPOCHS):

        svd_model.train()

        optimizer.zero_grad()

        preds = svd_model(
            train_users_svd,
            train_movies_svd
        )

        loss = loss_fn(
            preds,
            train_ratings_svd
        )

        loss.backward()

        optimizer.step()

    elapsed = (
        time.perf_counter()
        - start_time
    )

    # --------------------------------------------------------
    # Stop monitor
    # --------------------------------------------------------

    monitor.stop()

    result = monitor.get_results()

    result["run"] = run
    result["training_time_s"] = elapsed

    svd_cpu_runs.append(
        result
    )

    print(
        f"Training time: {elapsed:.4f} s"
    )


# ============================================================
# SAVE
# ============================================================

svd_cpu_file = os.path.join(
    RESULT_DIR,
    "FunkSVD_CPU_hardware_results.csv"
)

svd_cpu_results_df = (
    save_five_run_results(
        svd_cpu_runs,
        svd_cpu_file,
        "FunkSVD",
        "CPU"
    )
)


========================= FunkSVD CPU RUN 1/5 =========================
Training time: 3.0298 s

========================= FunkSVD CPU RUN 2/5 =========================
Training time: 2.3178 s

========================= FunkSVD CPU RUN 3/5 =========================
Training time: 1.8554 s

========================= FunkSVD CPU RUN 4/5 =========================
Training time: 1.8630 s

========================= FunkSVD CPU RUN 5/5 =========================
Training time: 1.9389 s

FunkSVD - CPU


,avg_cpu_util_percent,peak_cpu_util_percent,avg_ram_mb,peak_ram_mb,avg_gpu_util_percent,peak_gpu_util_percent,avg_gpu_memory_util_percent,peak_gpu_memory_util_percent,avg_vram_mb,peak_vram_mb,avg_gpu_power_w,peak_gpu_power_w,avg_gpu_temperature_c,peak_gpu_temperature_c,gpu_energy_joules,run,training_time_s,Algorithm,Device
0,45.269,52.000,919.676,945.609,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3.030,FunkSVD,CPU
1,193.683,1794.150,945.609,945.609,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2.318,FunkSVD,CPU
2,45.115,52.400,927.158,945.609,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,1.855,FunkSVD,CPU
3,44.640,52.400,913.310,930.223,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,1.863,FunkSVD,CPU
4,44.880,52.350,934.834,960.992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,1.939,FunkSVD,CPU
5,74.717,400.660,928.117,945.609,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AVERAGE,2.201,FunkSVD,CPU
6,66.504,778.985,12.674,10.879,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STD,0.501,FunkSVD,CPU



Saved result file:
/content/drive/MyDrive/GCMC_Project/hardware_results/FunkSVD_CPU_hardware_results.csv


In [ ]:
# Cell 16: Evaluate FunkSVD on the test set

test_users_svd = torch.tensor(
    test_df.userId.map(user_map).values,
    dtype=torch.long
)

test_movies_svd = torch.tensor(
    test_df.movieId.map(movie_map).values,
    dtype=torch.long
)

test_actual_svd = test_df.rating.values

svd_model.eval()
with torch.no_grad():
    preds_svd = svd_model(test_users_svd, test_movies_svd).numpy()

svd_rmse = np.sqrt(mean_squared_error(test_actual_svd, preds_svd))

print("===================")
print("FUNK SVD RESULTS")
print("===================")
print(f"RMSE = {svd_rmse:.4f}")

FUNK SVD RESULTS
RMSE = 2.7621


In [ ]:
# Cell 17: Model Comparison — GCMC vs FunkSVD

print("===================")
print("MODEL COMPARISON")
print("===================")
print(f"GCMC RMSE : {gcmc_rmse:.4f}")
print(f"SVD RMSE  : {svd_rmse:.4f}")

if gcmc_rmse < svd_rmse:
    print("\nWinner: GCMC")
else:
    print("\nWinner: SVD")

MODEL COMPARISON
GCMC RMSE : 1.0630
SVD RMSE  : 2.7621

Winner: GCMC
